# Практична робота №4
**Тема:** Нейронні мережі

## Мета роботи
Закріпити основні поняття використання нейронных мереж для розпізнавання зображень

Завдання:

1.	Розглянути задачу класифікації рукописних цифр з використанням набору даних MNIST. Це популярний набір даних, який містить зображення рукописних цифр (від 0 до 9). Створити просту модель нейронної мережі для розпізнавання цих цифр.




**Крок 1.** Імпортуйте необхідні бібліотеки

In [1]:
# Робота з даними
import numpy as np
import pandas as pd

# Візуалізація
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow та Keras для нейронних мереж
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Побудова графіків та діаграм
import matplotlib.pyplot as plt

# Метрики та інструменти
from sklearn.metrics import classification_report, confusion_matrix

**Крок 2**. Завантажте дані. Це можна зробити безпосередньо з бібліотеки Keras

Що ми отримуємо:

x_train → 60 000 зображень для навчання (28×28 пікселів).

y_train → відповідні мітки (цифри від 0 до 9).

x_test → 10 000 зображень для перевірки.

y_test → мітки для тестових зображень.

In [ ]:
# Завантаження набору даних MNIST
from tensorflow.keras.datasets import mnist

# Розділення на тренувальні та тестові дані
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Перевіримо розміри масивів
print("Розмір тренувальних даних:", x_train.shape)
print("Розмір тестових даних:", x_test.shape)

**Крок 3.** Візуалізуємо кілька прикладів цифр, щоб переконатися, що дані завантажені правильно.

In [ ]:
# Відображення перших 9 зображень із тренувального набору
plt.figure(figsize=(8,8))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(x_train[i], cmap='gray')
    plt.title(f"Цифра: {y_train[i]}")
    plt.axis('off')
plt.show()

**Крок 4. Підготовка даних.**

1. Потрібно обробити картинки так, щоб значення кожного піксела було чітко чорно-білим, могло бути описано 0 або 1.
2. Потрібно представити мітки у вигляді векторів, наприклад 7 має бути замінено на вектор [0,0,0,0,0,0,0,1,0,0].

In [ ]:
# Нормалізація пікселів: значення від 0 до 255 → від 0 до 1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Перетворення міток у формат one-hot
num_classes = 10
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

print("Форма x_train:", x_train.shape)
print("Форма y_train:", y_train.shape)

**Крок 5** Побудова нейронної мережі для розпізнавання цифр

Пояснення

Flatten → розгортає матрицю 28×28 у вектор довжиною 784.

Dense(128, relu) → прихований шар із функцією активації ReLU.

Dense(64, relu) → ще один прихований шар для більшої виразності.

Dense(10, softmax) → вихідний шар, який дає ймовірності для кожної цифри (0–9).

Adam + categorical_crossentropy → стандартна комбінація для багатокласової класифікації.

**ReLU (Rectified Linear Unit)**- Функція активації, яка працює так:
Формула:𝑓(𝑥)=mаx(0,𝑥). Якщо вхідне значення додатне → воно проходить без змін.Якщо негативне → замінюється на 0.

**Softmax** - Формула, яка перетворює набір чисел (логіти) у ймовірності, що сумуються до 1. Наприклад, якщо мережа бачить цифру "7", вона може видати:

0 → 0.01, 1 → 0.00, …, 7 → 0.95, 8 → 0.02, 9 → 0.02.

Найбільша ймовірність → передбачений клас.




In [ ]:
model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),       # Перетворюємо 28x28 пікселі у вектор
    layers.Dense(256, activation='relu'),       # Прихований шар з 128 нейронами
    layers.Dense(64, activation='relu'),        # Ще один прихований шар
    layers.Dense(10, activation='softmax')      # Вихідний шар (10 класів: цифри 0-9)
])

# Компіляція моделі
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Переглянемо архітектуру
model.summary()


**Крок 6.Навчаємо модель на тренувальних даних**
Використаємо метод fit, щоб навчати мережу кілька епох і паралельно перевіряти точність на тестових даних.

Пояснення параметрів

epochs=10 → модель пройде тренувальні дані 10 разів.

batch_size=128 → дані подаються невеликими порціями по 128 зображень.

validation_data → дозволяє бачити точність на тестових даних після кожної епохи.

history → зберігає інформацію про точність і втрати, щоб ми могли побудувати графіки.


In [ ]:
# Навчання моделі
history = model.fit(
    x_train, y_train,                # тренувальні дані
    validation_data=(x_test, y_test),# перевірка на тестових даних
    epochs=10,                       # кількість епох (повних проходів по даних)
    batch_size=128,                  # розмір пакету
    verbose=2                        # рівень деталізації
)

**Крок 6**. Оцінюємо якість моделі

Точність: як добре модель класифікує цифри на тренувальних і тестових даних.

Втрати (loss): наскільки модель помиляється; чим менше — тим краще.

Якщо графіки тренувальних і валідаційних кривих йдуть поруч → модель узагальнює добре.

Якщо тренувальна точність значно вища за тестову → є ризик перенавчання.

In [ ]:
# Оцінка моделі на тестових даних
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Точність на тестових даних: {test_acc:.4f}")
print(f"Втрати на тестових даних: {test_loss:.4f}")
y_pred_1=model.predict(x_test)
y_pred = np.argmax(y_pred_1, axis=1)
y_test_labels = np.argmax(y_test, axis=1)
print(f'Матриця плутанини:{confusion_matrix(y_test_labels,y_pred)}')

# Побудова графіків точності та втрат
plt.figure(figsize=(12,5))

# Графік точності
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Тренувальна точність')
plt.plot(history.history['val_accuracy'], label='Валідаційна точність')
plt.title('Точність моделі')
plt.xlabel('Епоха')
plt.ylabel('Точність')
plt.legend()

# Графік втрат
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Тренувальні втрати')
plt.plot(history.history['val_loss'], label='Валідаційні втрати')
plt.title('Втрати моделі')
plt.xlabel('Епоха')
plt.ylabel('Втрати')
plt.legend()

plt.show()

**Крок 7** Зробимо прогноз для кількох випадкових зображень і порівняємо з правильними мітками.

In [ ]:
# Передбачення для перших 9 зображень із тестового набору
predictions = model.predict(x_test)

plt.figure(figsize=(8,8))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(x_test[i], cmap='gray')
    predicted_label = np.argmax(predictions[i])   # клас з найбільшою ймовірністю
    true_label = np.argmax(y_test[i])             # правильна мітка
    plt.title(f"Прогноз: {predicted_label}, Правильно: {true_label}")
    plt.axis('off')

**Завдання для самостійного виконання**

1. Модифікація архітектури моделі

Змінити кількість шарів або нейронів у прихованих шарах.

Порівняти точність і втрати при різних архітектурах.

2. Експерименти з функціями активації

Спробувати замінити ReLU на sigmoid або tanh.

Пояснити, як це вплинуло на результат.

3. Робота з параметрами навчання

Змінити кількість епох, розмір batch_size, оптимізатор (Adam → SGD).

Порівняти швидкість і якість навчання.

4. Дослідження матриці неточностей (confusion matrix)

перед побудовою перетворіть прогнозні і тестові дані знову в мітки

y_pred = np.argmax(y_pred_probs, axis=1)

Побудувати матрицю й пояснити, які класи найскладніші для моделі.

Рядки (Rows) — Реальні значення. Кожен рядок показує, як нейромережа класифікувала всі об'єкти конкретного класу, що були в тестовій вибірці.

Приклад (Рядок 0):

У вашому наборі було 980 нулів (970+1+2+2+1+1+3).

З них модель впізнала 970 як "0", а 10 — переплутала з іншими цифрами.

Стовпчики (Columns) — Передбачення (Predicted). Кожен стовпчик показує, скільки разів модель видала певний результат, і чи був він правильним.

Приклад (Стовпчик 1):

Модель сказала "це одиниця" 1144 рази (0+1129+2+1+1+1+3+3+1+4). Але насправді одиницями були лише 1129 з них.

In [ ]:
#Тут має бути Ваш код


Тут мають бути Ваші висновки